In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [2]:
df=pd.read_csv('../data/tested.csv')

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [5]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [6]:
df.isnull().sum()

Survived     0
Pclass       0
Sex          0
Age         86
SibSp        0
Parch        0
Fare         1
Embarked     0
dtype: int64

# Step1-Train/test/split
# for Pipeline:
### 1. (CLT)Impute Missing for age and fare 
### 2. (CLT)OneHotEncoding for sex and embarked
### 3. (CLT)Scaling(MinMax because of feature selection )
### 4.   Feature selection
### 5.    Model Training

In [7]:
X_train,X_test,y_train,y_test=train_test_split(
    df.drop(columns=['Survived']),
    df['Survived'],
    test_size=0.2,
    random_state=42
)

In [8]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
336,2,male,32.0,0,0,13.0000,S
31,2,male,24.0,2,0,31.5000,S
84,2,male,NaN,0,0,10.7083,Q
287,1,male,24.0,1,0,82.2667,S
317,2,male,19.0,0,0,10.5000,S


In [9]:
trf1= ColumnTransformer([
    ("impute_age",SimpleImputer(),[2]),
    ("impute_fare",SimpleImputer(),[5])
],remainder="passthrough")

In [18]:
trf2= ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[3,6])
],remainder='passthrough'
                       )
# Handleunknown ignore will encode the unknown values as all zeros,

In [19]:
trf3=ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,8)) #slice apply transformation in all the columns provided to it
],remainder='passthrough') 

In [20]:
trf4=SelectKBest(score_func=chi2,k=8)

In [21]:
trf5=DecisionTreeClassifier()

# Creating the pipeline

In [48]:
pipe=Pipeline([
             ('trf1',trf1),
             ('trf2',trf2),
             ('trf3',trf3),
             ('trf4',trf4),
             ('trf5',trf5)])

# make_pipeline

In [28]:
# pipe=make_pipeline(trf1,trf2,trf3,trf4,trf5)

# Train

In [49]:
pipe.fit(X_train,y_train)

,steps,"[('trf1', ...), ('trf2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('impute_fare', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [34]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_

array([30.11576336])

In [35]:
pipe.named_steps['trf1'].transformers_[1][1].statistics_

array([36.90913514])

# Predict

In [36]:
y_pred= pipe.predict(X_test)

In [37]:
y_pred

array([0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0,
       0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1])

In [38]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

1.0

In [39]:


pipe.fit(X_train, y_train)

y_train_pred = pipe.predict(X_train)
y_test_pred = pipe.predict(X_test)

print("Training accuracy:", accuracy_score(y_train, y_train_pred))
print("Testing accuracy:", accuracy_score(y_test, y_test_pred))
print("Test samples:", len(y_test))
print("Correct test predictions:", sum(y_test == y_test_pred))

Training accuracy: 1.0
Testing accuracy: 1.0
Test samples: 84
Correct test predictions: 84


In [40]:
# Checking the shape of the data after each transformation
X1 = trf1.fit_transform(X_train)
print("After trf1:", X1.shape)

X2 = trf2.fit_transform(X1)
print("After trf2:", X2.shape)

X3 = trf3.fit_transform(X2)
print("After trf3:", X3.shape)

X4 = trf4.fit_transform(X3, y_train)
print("After trf4:", X4.shape)

After trf1: (334, 7)
After trf2: (334, 10)
After trf3: (334, 10)
After trf4: (334, 8)


In [41]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("Target in X_train:", "Survived" in X_train.columns)
print("Target in X_test:", "Survived" in X_test.columns)

print("Train-test overlap:", len(set(X_train.index) & set(X_test.index)))

X_train shape: (334, 7)
X_test shape: (84, 7)
Target in X_train: False
Target in X_test: False
Train-test overlap: 0


# Cross Validation

In [42]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe,X_train,y_train,cv=5,scoring='accuracy').mean()

np.float64(1.0)

# Hyperparameter tuning 

- gridsearch using pipeline

In [43]:
params={
    'trf5__max_depth':[1,2,3,4,5,None]
}

In [44]:
from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(pipe,params,cv=5,scoring='accuracy')
grid.fit(X_train,y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'trf5__max_depth': [1, 2, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('impute_age', ...), ('impute_fare', ...)]"


In [45]:
grid.best_score_

np.float64(1.0)

In [46]:
grid.best_params_

{'trf5__max_depth': 1}

# Exporting the pipeline

In [51]:
import pickle 
pickle.dump(pipe,open('../data/pipe.pkl','wb'))